In [6]:
# Install packages into the Codespaces environment
!pip install pandas requests beautifulsoup4

import time
from bs4 import BeautifulSoup
import pandas as pd
import requests

headers = {
    'User-Agent': 'Mozilla/5.0 (Educational purpose scraper - ISYS5002)',
    'Accept': 'text/html,application/xhtml+xml',
}

print("Setup complete!")

Setup complete!


In [7]:
# Cell 2: Download the Web Page
url = 'http://books.toscrape.com/'

# Send HTTP GET request
response = requests.get(url, headers=headers)

# Check if request was successful
if response.status_code == 200:
  print(
      f"Successfully downloaded the page! Content length:"
      f" {len(response.text)} characters"
  )
else:
  print(f"Failed to download the page. Status code: {response.status_code}")

# View first 300 characters of raw HTML
print("\nHTML Preview:")
print(response.text[:300])

Successfully downloaded the page! Content length: 51294 characters

HTML Preview:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lan


In [8]:
# Cell 3: Parse HTML with BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')

# Extract and print the page title
page_title = soup.title.text
print(f"Page title: {page_title}")

Page title: 
    All products | Books to Scrape - Sandbox



In [9]:
# Cell 4: Extract Book Information
book_containers = soup.find_all('article', class_='product_pod')
print(f"Found {len(book_containers)} books on this page.")

titles = []
prices = []
ratings = []

for book in book_containers:
    # Extract title
    title = book.h3.a['title']
    titles.append(title)
    
    # Extract price
    price = book.find('p', class_='price_color').text
    prices.append(price)
    
    # Extract star rating
    star_rating = book.find('p', class_='star-rating')['class'][1]
    ratings.append(star_rating)

# Assemble initial DataFrame
books_df = pd.DataFrame({
    'Title': titles,
    'Price': prices,
    'Rating': ratings
})

# Display first 5 rows
books_df.head()

Found 20 books on this page.


,Title,Price,Rating
0,A Light in the Attic,Â£51.77,Three
1,Tipping the Velvet,Â£53.74,One
2,Soumission,Â£50.10,One
3,Sharp Objects,Â£47.82,Four
4,Sapiens: A Brief History of Humankind,Â£54.23,Five


In [10]:
# Cell 5: Clean and Process Data

# Clean price: remove currency characters and convert to float
books_df['Price'] = (
    books_df['Price'].str.replace('Â', '').str.replace('£', '').astype(float)
)

# Map string ratings to numeric values (1 to 5)
rating_mapping = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5,
}
books_df['Rating'] = books_df['Rating'].map(rating_mapping)

# Display cleaned DataFrame preview
print("Cleaned Data Preview:")
display(books_df.head())

# Basic summary statistics
print("\nBasic Statistics:")
display(books_df.describe())

# Calculate average price grouped by rating
print("\nAverage Price by Rating:")
avg_price_by_rating = books_df.groupby('Rating')['Price'].mean().sort_index()
display(avg_price_by_rating)

Cleaned Data Preview:


,Title,Price,Rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5



Basic Statistics:


,Price,Rating
count,20.000000,20.000000
mean,38.048500,2.850000
std,15.135231,1.565248
min,13.990000,1.000000
25%,22.637500,1.000000
50%,41.380000,3.000000
75%,51.865000,4.000000
max,57.250000,5.000000



Average Price by Rating:


Rating
1    40.018333
2    36.830000
3    42.316667
4    31.105000
5    39.750000
Name: Price, dtype: float64

In [11]:
# Cell 6: Scrape Multiple Pages (Pages 1 to 3)

all_titles = []
all_prices = []
all_ratings = []

# Loop through pages 1, 2, and 3
for page_num in range(1, 4):
  if page_num == 1:
    page_url = 'http://books.toscrape.com/'
  else:
    page_url = f'http://books.toscrape.com/catalogue/page-{page_num}.html'

  # Add a 1-second delay for polite scraping
  time.sleep(1)

  resp = requests.get(page_url, headers=headers)

  if resp.status_code == 200:
    print(f"Successfully scraped page {page_num}")
    p_soup = BeautifulSoup(resp.text, 'html.parser')
    containers = p_soup.find_all('article', class_='product_pod')

    for book in containers:
      all_titles.append(book.h3.a['title'])
      all_prices.append(book.find('p', class_='price_color').text)
      all_ratings.append(book.find('p', class_='star-rating')['class'][1])
  else:
    print(f"Failed to scrape page {page_num}")

# Assemble multi-page DataFrame
all_books_df = pd.DataFrame({
    'Title': all_titles,
    'Price': all_prices,
    'Rating': all_ratings,
})

# Clean multi-page dataset
all_books_df['Price'] = (
    all_books_df['Price']
    .str.replace('Â', '')
    .str.replace('£', '')
    .astype(float)
)
all_books_df['Rating'] = all_books_df['Rating'].map(rating_mapping)

print(f"\nTotal books scraped across 3 pages: {len(all_books_df)}")
display(all_books_df.head(10))

Successfully scraped page 1
Successfully scraped page 2
Successfully scraped page 3

Total books scraped across 3 pages: 60


,Title,Price,Rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5
5,The Requiem Red,22.65,1
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4
7,The Coming Woman: A Novel Based on the Life of...,17.93,3
8,The Boys in the Boat: Nine Americans and Their...,22.60,4
9,The Black Maria,52.15,1


In [13]:
# Cell 7: Save Scraped Data to CSV File
all_books_df.to_csv('Week10-Scrapped Books.csv', index=False)
print(
    "Data successfully saved to 'Week10-Scrapped Books.csv' in your"
    " workspace!"
)

Data successfully saved to 'Week10-Scrapped Books.csv' in your workspace!
